### Imports

In [199]:
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import pyopencl as cl
import time

### 8a RGB-Bild einlesen und in ein Graustufenbild umwandeln + 
### 8b Helligkeit und Kontrast vom Graustufenbild anpassen

In [ ]:
# Only one path should be active at a time so different image sizes can be tested

image_path = Path("images_input/1.nature_small.jpeg")
## image_path = Path("images_input/2.nature_medium.jpeg")
## image_path = Path("images_input/3.nature_large.jpeg")
## image_path = Path("images_input/4.nature_mega.jpeg")

# uint8 is used because RGB values are stored in the range 0-255
image = Image.open(image_path)
rgb_matrix = np.array(image, dtype=np.uint8)

# RGB images have the shape: height x width x 3 color channels
height, width, channels = rgb_matrix.shape
# This value is also used as the number of OpenCL work-items
pixel_count = height * width

# Convert the 3D RGB matrix into a one-dimensional array
# This representation makes indexing inside the OpenCL kernel simple
rgb_flat = rgb_matrix.flatten()
gray_matrix_flat = np.empty(pixel_count, dtype=np.uint8)
# Create a histogram with 256 entries
histogram = np.zeros(256, dtype=np.int32)

# Parameters for contrast and brightness adjustment
# alpha controls contrast, beta controls brightness
# float32 is used because OpenCL's float datatype is 32-bit
alpha = np.float32(1.3)
beta = np.float32(20)

# The context represents the OpenCL device that will execute the kernels, for example the GPU
context = cl.create_some_context()

# Kernel executions and memory transfers are submitted to the device through this queue
queue = cl.CommandQueue(context)

# Shortcut for OpenCL memory flags
mf = cl.mem_flags


# Create the input buffer and copy the flattened RGB image from the CPU memory to the OpenCL device
rgb_input_buffer = cl.Buffer(context, mf.READ_ONLY | mf.COPY_HOST_PTR, hostbuf=rgb_flat)
# Allocate an output buffer for the processed grayscale image & The kernel writes the final grayscale/brightness/contrast values into this buffer
gray_output_buffer = cl.Buffer(context, mf.WRITE_ONLY, gray_matrix_flat.nbytes)
# Create the histogram buffer and initialize it with zeros
# READ_WRITE is required because the histogram kernel increments its entries
histogram_buffer = cl.Buffer(context, mf.READ_WRITE | mf.COPY_HOST_PTR, hostbuf=histogram)

# The OpenCL program containing both kernels
program = cl.Program(context, """
#pragma OPENCL EXTENSION cl_khr_global_int32_base_atomics : enable
/*
 * Kernel 1:
 * Converts the RGB image to grayscale and directly applies
 * contrast and brightness adjustment.
 *
 * Each OpenCL work-item processes exactly one pixel.
 */
__kernel void rgb_to_grayscale_brightness_contrast(
    __global const uchar *rgb,
    __global uchar *gray_out,
    const float alpha,
    const float beta
)
{
    // This ID corresponds to one pixel in the image                 
    int pixel_id = get_global_id(0);
    // Every RGB pixel consists of three consecutive values                 
    int rgb_id = pixel_id * 3;

    uchar red = rgb[rgb_id];
    uchar green = rgb[rgb_id + 1];
    uchar blue = rgb[rgb_id + 2];

    // Convert RGB to grayscale using the weighting defined in the task                 
    uchar gray_value = (uchar)(0.21f * red + 0.72f * green + 0.07f * blue);
    // Apply contrast and brightness
    float value = alpha * gray_value + beta;

    // Clamp values below 0 to the valid minimum & maximum grayscale value                 
    if (value < 0.0f) {
        value = 0.0f;
    }

    if (value > 255.0f) {
        value = 255.0f;
    }

    gray_out[pixel_id] = (uchar)(value);
}
/*
 * Kernel 2:
 * Calculates the histogram of the processed grayscale image.
 *
 * Each work-item reads one grayscale pixel and increments
 * the corresponding histogram entry.
 */                     
__kernel void compute_histogram(
    __global const uchar *gray,
    __global int *histogram
)
{
    int pixel_id = get_global_id(0);
    uchar value = gray[pixel_id];
                     
    /*
     * Several pixels may have the same grayscale value and therefore
     * attempt to update the same histogram entry simultaneously.
     *
     * atomic_inc prevents race conditions by ensuring that each
     * increment operation is executed safely.
     */                 
    atomic_inc(&histogram[value]);
}
""").build()


start = time.perf_counter()

# Execute the first kernel
program.rgb_to_grayscale_brightness_contrast(
    queue,
    (pixel_count,),
    None,
    rgb_input_buffer,
    gray_output_buffer,
    alpha,
    beta
)

# Execute the histogram kernel
program.compute_histogram(
    queue,
    (pixel_count,),
    None,
    gray_output_buffer,
    histogram_buffer
)

# Copy the processed grayscale image from GPU/device memory back into the NumPy array in CPU memory
cl.enqueue_copy(queue, gray_matrix_flat, gray_output_buffer)
# Copy the completed histogram back to CPU memory
cl.enqueue_copy(queue, histogram, histogram_buffer)
queue.finish()

# Convert the flattened grayscale result back into the original two-dimensional image shape
gray_matrix_2 = gray_matrix_flat.reshape((height, width))

end = time.perf_counter()

# Calculate and display the total measured PyOpenCL runtime
pyopencl_time = end - start
print(pyopencl_time)

# Optional visualization of the resulting grayscale image
## gray_matrix = gray_flat.reshape((height, width))
## plt.imshow(gray_matrix, cmap="gray", vmin=0, vmax=255)
## plt.axis("off")
## plt.show()

0.0027275001630187035


### 8c Histogramm der Grauwerte berechnen und darstellen

In [ ]:
# optionale Darstellung des Histogramms; sie bleibt auskommentiert, damit Plot-Erzeugung und Messung der Verarbeitung getrennt bleiben

## histogram = np.bincount(gray_matrix_2.flatten(), minlength=256)

# plt.figure(figsize=(10, 5))
# plt.bar(range(256), histogram, width=1.0)
# plt.xlabel("Grauwert")
# plt.ylabel("Anzahl Pixel")
# plt.title("Histogramm der Grauwerte")
# plt.xlim(0, 256)
# plt.grid(axis="y", alpha=0.3)
# plt.show()